# 진상 발언 감지 AI 학습 (Gallio)

음성인식으로 받은 문장이 **공격적인 말(욕설·하대·갑질·위협)**인지 판단하는 AI를 학습해서, 워치에 넣을 `text_model.tsv` 파일을 만듭니다.

**순서:** 위에서부터 셀을 하나씩 실행 (▶ 또는 Shift+Enter) → 마지막에 `text_model.tsv` 다운로드 → Android Studio의 `app/src/main/assets` 폴더에 덮어쓰기

**데이터**
- 공개 데이터: Smilegate UnSmile (CC BY-NC-ND 4.0), Korean HateSpeech Dataset (CC BY-SA 4.0), 2runo Curse-detection-data (MIT), ZIZUN korean-malicious-comments-dataset (MIT) — 비상업·학술 목적으로만 사용, 발표 자료에 출처 표기
- 매장 진상 문장: 팀이 직접 작성 (`jinsang_seed.csv`) + **팀이 모은 문장** (`team_jinsang.csv`, 선택)

**방식:** 1~3글자 조각(n-gram) TF-IDF + 로지스틱 회귀. 워치에서 별도 라이브러리 없이 계산할 수 있게 일부러 단순한 모델을 씁니다.

## 1. 공개 데이터 받기

In [ ]:
!git clone -q --depth 1 https://github.com/smilegate-ai/korean_unsmile_dataset.git data/korean_unsmile_dataset
!git clone -q --depth 1 https://github.com/kocohub/korean-hate-speech.git data/korean-hate-speech
!git clone -q --depth 1 https://github.com/2runo/Curse-detection-data.git data/Curse-detection-data
!git clone -q --depth 1 https://github.com/ZIZUN/korean-malicious-comments-dataset.git data/korean-malicious-comments-dataset
!ls data

## 2. 매장 진상 시드 문장 (기본)

`split`이 `test`인 문장은 **학습에 쓰지 않고** 정확도 측정에만 씁니다. 여기에 문장을 추가해도 되고, 아래 3번처럼 별도 파일로 올려도 됩니다.

In [ ]:
%%writefile jinsang_seed.csv
text,label,split
사장 나오라고 해,1,train
사장 불러와 당장,1,train
본사에 전화할 거야 각오해,1,train
본사에 민원 넣을 거니까 그렇게 알아,1,train
내가 누군지 알아,1,train
내가 여기 단골인데 이따위로 해,1,train
손님이 왕인 거 몰라,1,train
고객센터에 다 말할 거야 너 이름 뭐야,1,train
너 잘리게 해줄까,1,train
너 오늘 여기서 잘리고 싶어,1,train
점장 번호 내놔,1,train
명찰 보여봐 신고할 거니까,1,train
리뷰 테러 해버린다,1,train
별점 일점 줄 테니까 각오해,1,train
알바 주제에 어디서 말대꾸야,1,train
니가 뭔데 나한테 이래라저래라야,1,train
너 몇 살이야 어린 게,1,train
부모가 그렇게 가르쳤냐,1,train
그것도 못해 머리가 나쁘냐,1,train
그러니까 알바나 하고 있지,1,train
말귀를 못 알아듣네 진짜,1,train
이거 하나 제대로 못 해 어,1,train
어린 게 싸가지가 없네,1,train
어디서 눈을 그렇게 떠,1,train
표정이 그게 뭐야 기분 나쁘게,1,train
웃어 웃으라고 손님한테,1,train
대답 안 해 귀 먹었어,1,train
답답해서 같이 일 못 하겠네 너 일 그렇게 할 거야,1,train
너 같은 애들 때문에 나라가 이 모양이야,1,train
배운 게 없으니까 이러지,1,train
가정교육을 어떻게 받은 거야,1,train
밖에서 보자,1,train
얼굴 기억했다,1,train
두고 봐 가만 안 둔다,1,train
퇴근할 때 보자 너,1,train
너 몇 시에 끝나 기다릴게,1,train
확 때려버릴까 보다,1,train
한 대 맞고 싶어,1,train
죽고 싶어 환장했냐,1,train
집이 어디야 찾아간다,1,train
여기 다 부숴버린다,1,train
이거 던져버린다 진짜,1,train
이 씨발 뭐 하는 거야,1,train
병신같이 일하네,1,train
미친년아 빨리 해,1,train
개새끼가 어디서 손님한테,1,train
지랄하고 있네,1,train
꺼져 꼴도 보기 싫어,1,train
닥치고 환불이나 해,1,train
좆같이 일하네 진짜,1,train
또라이 아니야 이거,1,train
야 빨리빨리 안 해,1,train
야 너 이리 와봐,1,train
야 알바 이거 다시 해와,1,train
돈 여기 있으니까 주워,1,train
카드 던져줄 테니까 받아,1,train
어디서 감히 손님을 가르치려고 들어,1,train
너 지금 나 무시하냐,1,train
내가 우습냐 지금,1,train
너 지금 나랑 싸우자는 거야,1,train
목소리 깔지 마라,1,train
눈 깔아,1,train
한 번만 더 대들어 봐,1,train
이게 사람이 할 짓이야 너 미쳤어,1,train
쓰레기 같은 가게네 직원도 쓰레기고,1,train
너 인생 그렇게 살지 마라,1,train
니네 사장도 너 같은 거 쓰는 거 보면 뻔하다,1,train
경찰 불러 볼래 내가 무서울 거 같아,1,train
멍청하냐 계산을 왜 이렇게 해,1,train
무릎 꿇고 사과해,1,train
당장 사과 안 하면 가만 안 있어,1,train
손님한테 말버릇이 그게 뭐야,1,train
너 때문에 기분 다 잡쳤어 책임져,1,train
어이 거기 알바 귀 안 들려,1,train
머리에 뭐가 들었냐,1,train
돈 받고 일하면 똑바로 해야지 이 새끼야,1,train
너 오늘 집에 못 갈 줄 알아,1,train
뒤질래 진짜,1,train
너 지금 반말했냐,1,train
야 이 미친놈아 이게 몇 번째야,1,train
어서 오세요,0,train
봉투 필요하세요,0,train
포인트 적립해 드릴까요,0,train
카드 꽂아주세요,0,train
영수증 드릴까요,0,train
이거 얼마예요,0,train
화장실 어디예요,0,train
따뜻한 아메리카노 한 잔 주세요,0,train
아이스로 주세요,0,train
포장해 주세요,0,train
여기서 먹고 갈게요,0,train
이거 유통기한 지난 것 같은데요,0,train
주문한 거랑 다르게 나왔어요,0,train
이거 환불 되나요,0,train
교환하고 싶은데 영수증 여기 있어요,0,train
계산이 잘못된 것 같아요 한 번만 확인해 주세요,0,train
기다린 지 좀 됐는데 언제 나와요,0,train
죄송한데 좀 빨리 부탁드려요,0,train
와 배고파 죽겠다,0,train
더워 죽겠네 오늘,0,train
웃겨 죽겠다 진짜,0,train
이거 진짜 미쳤다 맛있다,0,train
대박 이거 신상이에요,0,train
사장님 오늘 안 나오셨어요,0,train
본사 이벤트 아직 해요,0,train
명찰 귀엽네요,0,train
몇 시까지 영업해요,0,train
고생 많으세요,0,train
감사합니다 수고하세요,0,train
잔돈은 괜찮아요,0,train
삼만 원 받았습니다,0,train
적립 번호 공일공 일이삼사,0,train
이거 두 개 사면 하나 더 주는 거죠,0,train
담배 하나 주세요,0,train
신분증 보여주시겠어요,0,train
죄송합니다 저희가 확인해 드릴게요,0,train
잠시만 기다려 주세요,0,train
다음 손님 도와드릴게요,0,train
혹시 이거 재고 더 있어요,0,train
택배 접수하려고요,0,train
칼국수 하나 주세요,0,train
칼로리 낮은 거 있어요,0,train
전복죽 하나 포장이요,0,train
때마침 잘 왔네 세일하네,0,train
야 이거 맛있다 너도 먹어봐,0,train
야 빨리 와 여기 자리 있어,0,train
엄마 이거 사줘,0,train
결제는 삼성페이로 할게요,0,train
현금영수증 해주세요,0,train
비닐 말고 종이봉투 있어요,0,train
얼음 조금만 넣어 주세요,0,train
샷 추가해 주세요,0,train
이거 데워 주실 수 있어요,0,train
전자레인지 써도 돼요,0,train
죄송한데 이거 하나 바꿔도 될까요,0,train
아까 계산할 때 할인이 안 들어간 것 같아요,0,train
제가 주문한 게 아직 안 나왔어요,0,train
혹시 와이파이 비밀번호 알려주실 수 있나요,0,train
주차 등록해 주세요,0,train
이거 사이즈 큰 거 있나요,0,train
입어봐도 돼요,0,train
이 제품 어떤 게 더 잘 나가요,0,train
설명 좀 해주실 수 있어요,0,train
할부 되나요 삼 개월로 해주세요,0,train
아 진짜 너무 늦게 나와서 짜증 나네요 다음엔 빨리 부탁해요,0,train
조금 불친절하신 것 같아서 기분이 안 좋네요,0,train
솔직히 좀 실망했어요 그래도 수고하세요,0,train
환불 규정이 어떻게 되는지 알려주세요,0,train
점장님 계시면 잠깐 이야기할 수 있을까요,0,train
네 알겠습니다 다음에 올게요,0,train
수고 많으십니다 날씨 춥죠,0,train
늦은 시간까지 고생하시네요,0,train
죽이는데 이 노래 뭐야,0,train
너 오늘 몇 시에 끝나 같이 밥 먹자,0,train
여기 진짜 싸다 미쳤다,0,train
니가 골라 나는 아무거나,0,train
이거 맛없으면 너 책임져 하하,0,train
어디서 감히 손님한테 대들어,1,test
너 내일부터 출근 못 하게 해줄게,1,test
이렇게 일할 거면 때려치워,1,test
니 얼굴 찍어서 인터넷에 올린다,1,test
부모님은 뭐 하시냐 이렇게 키우고,1,test
무식하게 생겨가지고 말도 안 통하네,1,test
골목에서 기다린다,1,test
멍청하게 그것도 계산 못하냐,1,test
한 번만 더 그러면 가만 안 있는다,1,test
싸가지 없는 게 어디서 말대꾸야,1,test
내 돈 내고 왜 이런 대접 받아야 돼 사장 나와,1,test
확 엎어버릴라 진짜,1,test
니가 책임질 거야 어 대답해,1,test
너 같은 게 무슨 서비스를 한다고,1,test
이딴 식으로 하면 장사 접게 만들어 준다,1,test
너 밤길 조심해라,1,test
시끄러 이 씨발년아,1,test
병신 새끼 일 하나 제대로 못 하고,1,test
니 월급 내가 주는 거야 알아,1,test
아 열받네 진짜 너 이리 나와,1,test
어린놈이 어디서 눈을 부라려,1,test
머리 나쁜 거 티 내지 마,1,test
너 일 몇 년 했어 이것도 몰라 바보야,1,test
나 여기 다시는 안 와 니네 다 망해라,1,test
꼴값 떨지 말고 빨리 해,1,test
너 죽고 싶냐,1,test
따라 나와 할 말 있어,1,test
경찰 부르든 말든 맘대로 해 너부터 혼나,1,test
지금 나한테 웃냐 비웃냐,1,test
사과 똑바로 해 고개 숙여,1,test
안녕하세요 주문하시겠어요,0,test
아메리카노 두 잔이랑 라떼 하나요,0,test
이거 포장 가능한가요,0,test
봉투 하나만 주세요,0,test
교통카드 충전해 주세요,0,test
이거 할인 상품 맞나요,0,test
계산 따로 해주세요,0,test
죄송한데 이거 반품하고 싶어요,0,test
음료가 너무 달아요 다시 만들어 주실 수 있나요,0,test
기다리는 동안 앉아 있어도 되죠,0,test
배고파 죽겠는데 빨리 되나요,0,test
여기 맛집이라더니 진짜 미쳤네요,0,test
점장님한테 칭찬 좀 전해 주세요,0,test
영수증은 버려 주세요,0,test
화장실 열쇠 주실 수 있어요,0,test
이 물건 어디 있는지 알려주실래요,0,test
가격이 좀 다르게 찍힌 것 같아요 확인 부탁드려요,0,test
불편하게 해드려서 죄송합니다,0,test
오늘 재고 언제 들어와요,0,test
카드 결제 오류 났는데 다시 해볼게요,0,test
너무 오래 걸려서 좀 속상하네요,0,test
수고하세요 좋은 하루 보내세요,0,test
칼 같은 거 파는 데 어디예요,0,test
이거 죽여주게 맛있네,0,test
야 너도 먹을래 내가 살게,0,test
사장님 바쁘시면 나중에 올게요,0,test
리뷰 좋게 남길게요,0,test
본사 고객센터 번호 알 수 있을까요,0,test
조금만 빨리 해주시면 감사하겠습니다,0,test
아 진짜 오늘 하루 힘들다,0,test
야 너 귀 안 들려 몇 번을 말해,1,train
내가 지금 몇 번째 말하는 줄 알아,1,train
너 지금 손님 무시하는 거지,1,train
일 똑바로 안 해 이 바보야,1,train
너 같은 걸 누가 뽑았어,1,train
사장한테 너 자르라고 할 거야,1,train
너 이름 적어갈 거야 각오해,1,train
여기 구청에 신고해서 문 닫게 해줄게,1,train
인터넷에 여기 다 올려버린다,1,train
너 오늘 제대로 걸렸다,1,train
어디서 쪼그만 게 대들어,1,train
나이도 어린 게 버르장머리 없이,1,train
엄마 아빠한테 그렇게 배웠어,1,train
대학은 나왔냐 이것도 모르게,1,train
공부 못해서 알바나 하는 주제에,1,train
그러니까 니가 이 나이 먹고 알바하지,1,train
얼굴도 못생긴 게 서비스도 엉망이네,1,train
너 생긴 거부터 마음에 안 들어,1,train
웃지 마 기분 나쁘니까,1,train
뭘 쳐다봐 눈 안 깔아,1,train
턱 들지 마라,1,train
입 다물고 시키는 대로 해,1,train
말 끊지 마 이게 어디서,1,train
한 번만 더 토 달면 혼난다,1,train
손님한테 반말하냐 지금,1,train
너 지금 나랑 해보자는 거냐,1,train
나 건드리면 큰일 나는 거 알지,1,train
우리 형이 누군지 알아,1,train
내가 아는 사람 많거든 너 하나쯤은 쉬워,1,train
너 이 동네에서 일 못 하게 만든다,1,train
밤에 혼자 다니지 마라,1,train
너 사는 데 알아낼 거야,1,train
뒤통수 조심해,1,train
확 그냥 패버릴라,1,train
멱살 잡히고 싶어,1,train
이거 얼굴에 던져버릴까,1,train
커피 확 부어버린다,1,train
물건 다 쓸어버린다 진짜,1,train
여기 박살 낸다,1,train
죽여버리고 싶네 진짜,1,train
너 진짜 맞아야 정신 차리지,1,train
야 이 새끼야 빨리 안 줘,1,train
씨발 장사 이따위로 할 거야,1,train
개같이 일하네 진짜,1,train
존나 느려 터졌네,1,train
미친 거 아니야 이게 뭐야,1,train
병신이냐 이것도 못 하게,1,train
지랄 말고 돈 돌려줘,1,train
꺼지라고 꼴 보기 싫으니까,1,train
닥쳐 변명하지 마,1,train
염병하네 진짜,1,train
썩을 것들,1,train
이 쌍놈의 가게,1,train
등신같이 서 있지 말고 움직여,1,train
머저리 같은 게,1,train
이 멍청아 거스름돈 틀렸잖아,1,train
돌대가리냐,1,train
한심하다 진짜 너 같은 애들,1,train
쓰레기 같은 서비스,1,train
아가씨 몇 살이야 번호 좀 줘,1,train
예쁜데 왜 이런 데서 일해 나랑 놀자,1,train
퇴근하고 나랑 술 한잔 하자니까 왜 거절해,1,train
손 좀 줘봐 잡아보게,1,train
웃으면서 서비스해야지 웃어 봐,1,train
너 여기서 일하는 거 니 부모가 알아,1,train
너 때문에 내 하루 망쳤어 돈으로 갚아,1,train
보상 안 해주면 여기서 안 나간다,1,train
너 무릎 꿇고 빌어,1,train
사과문 써서 가져와,1,train
돈 몇 푼 번다고 유세 떠냐,1,train
내 말이 말 같지 않아,1,train
니가 그렇게 잘났어,1,train
어디서 말을 그딴 식으로 해,1,train
하라면 하지 무슨 말이 많아,1,train
대가리 굴리지 마,1,train
너 지금 나 째려봤지,1,train
너 지금 한숨 쉬었냐 손님 앞에서,1,train
아 진짜 짜증 나게 하네 이 알바 새끼,1,train
너 같은 건 평생 알바나 해라,1,train
뭐 이런 게 다 있어,1,train
너 지금 일부러 이러는 거지,1,train
경찰 불러 봐 누가 이기나 보자,1,train
CCTV 있어도 상관없어 너 혼나 볼래,1,train
너희 사장 오기 전에 정신 차려,1,train
가만 두나 봐라,1,train
두고두고 찾아온다 너,1,train
너 집에 가서 두 다리 뻗고 자나 보자,1,train
어디 한번 더 해봐 확,1,train
손 올라가기 전에 똑바로 해,1,train
너 맞을래,1,train
입 찢어버린다,1,train
안녕하세요 몇 분이세요,0,train
두 명이요 창가 자리 있어요,0,train
메뉴판 좀 주세요,0,train
여기 제일 잘 나가는 게 뭐예요,0,train
맵기 조절 되나요,0,train
덜 맵게 해주세요,0,train
반찬 좀 더 주실 수 있어요,0,train
물은 셀프인가요,0,train
계산은 나가면서 할게요,0,train
각자 계산해도 돼요,0,train
여기 진짜 맛있네요 또 올게요,0,train
잘 먹었습니다,0,train
이거 남은 거 포장 돼요,0,train
라지 사이즈로 바꿔 주세요,0,train
디카페인 있어요,0,train
시럽 빼 주세요,0,train
휘핑 많이 올려 주세요,0,train
진동벨 주시는 거죠,0,train
이거 주문한 거 아닌데요 확인 부탁드려요,0,train
음료 하나가 빠진 것 같아요,0,train
빨대 어디 있어요,0,train
콘센트 쓸 수 있는 자리 있나요,0,train
몇 시까지 있어도 돼요,0,train
라면 끓여 먹어도 돼요,0,train
뜨거운 물 어디서 받아요,0,train
삼각김밥 이거 행사 상품이에요,0,train
도시락 데워 주세요,0,train
택배 보내려고 하는데 어떻게 해요,0,train
반값 택배 되나요,0,train
복권 한 장 주세요,0,train
교통카드 오만 원 충전이요,0,train
상품권 쓸 수 있어요,0,train
이거 이 사이즈 말고 한 치수 큰 거요,0,train
피팅룸 어디예요,0,train
이거 다른 색상 있나요,0,train
세일 언제까지 해요,0,train
교환하러 왔어요 영수증 여기요,0,train
태그 안 뗐는데 환불 되죠,0,train
이 폰 요금제 좀 설명해 주세요,0,train
약정 끝났는데 기기 바꾸려고요,0,train
할인 받을 수 있는 방법 있어요,0,train
생각해 보고 다시 올게요,0,train
설명 잘해주셔서 감사해요,0,train
화장품 샘플 좀 주실 수 있어요,0,train
이거 테스트해 봐도 돼요,0,train
피부가 민감한데 이거 괜찮을까요,0,train
PC방 자리 남은 거 있어요,0,train
두 시간 충전할게요,0,train
컴퓨터가 안 켜지는데 봐주실 수 있어요,0,train
마우스가 이상해요 바꿔 주세요,0,train
컵라면 하나랑 콜라요,0,train
아 진짜 오늘 비 엄청 오네요,0,train
퇴근 시간이라 사람 많네요,0,train
힘드시죠 화이팅하세요,0,train
수고하셨어요 좋은 저녁 되세요,0,train
아까는 제가 좀 예민했네요 죄송해요,0,train
제가 잘못 말했네요 다시 주문할게요,0,train
천천히 하셔도 돼요,0,train
바쁘신데 감사해요,0,train
여기 사장님 친절하시더라고요,0,train
죄송한데 이거 잘못 계산된 것 같아요,0,train
솔직히 좀 불편했어요 개선 부탁드려요,0,train
이건 좀 아닌 것 같아요 다시 확인해 주세요,0,train
기분이 좀 나쁘네요 그래도 괜찮아요,0,train
아까 직원분이 좀 퉁명스러우셨어요,0,train
환불 규정 문자로 보내 주실 수 있어요,0,train
매니저님 계시면 상담 좀 하고 싶어요,0,train
고객센터 번호 알려주세요 문의 좀 하게요,0,train
리뷰 이벤트 하고 갈게요,0,train
별점 다섯 개 드릴게요,0,train
아 배불러 죽겠다,0,train
피곤해 죽겠어 오늘,0,train
대박 미쳤다 이거 너무 귀엽다,0,train
야 이거 봐봐 완전 웃겨,0,train
너 이거 먹을 거야 반 나눠 먹자,0,train
야 빨리 골라 영화 시간 늦겠다,0,train
너 오늘 진짜 예쁘다,0,train
엄마 이거 사도 돼,0,train
아빠 나 저거 먹고 싶어,0,train
할머니 천천히 오세요,0,train
죽 한 그릇 포장이요,0,train
칼국수 곱빼기 되나요,0,train
때밀이 수건 어디 있어요,0,train
때 맞춰 잘 왔네 딱 문 열었네,0,train
패딩 이거 얼마예요,0,train
패스트푸드 말고 다른 거 먹자,0,train
죽이는 날씨다 오늘,0,train
이 노래 미쳤다 누구 노래예요,0,train
칼로리 표시 어디 있어요,0,train
신분증 확인 좀 하겠습니다,0,train
죄송합니다 바로 다시 만들어 드릴게요,0,train
불편을 드려 죄송합니다,0,train
잠시만요 확인해 보겠습니다,0,train
네 손님 말씀하세요,0,train
영수증 필요하시면 말씀해 주세요,0,train
포인트 카드 있으세요,0,train
현금이세요 카드세요,0,train
봉투 이십 원인데 괜찮으세요,0,train


## 3. (선택) 팀이 모은 진상 문장 올리기

알바하는 친구들에게 모은 "진상한테 들은 말"과 평범한 매장 대화를 CSV로 만들어 올리세요.

```
text,label,split
알바 주제에 어디서 말대꾸야,1,train
포인트 적립해 주세요,0,train
너 퇴근 몇 시야 기다린다,1,test
```
- `label`: 1 = 진상, 0 = 일반 (**일반 문장도 꼭 같이** 넣어야 오탐이 줄어요. 정당한 항의도 0으로!)
- `split`: train(학습) / test(평가). 10개 중 2개 정도는 test로 두면 정확도를 정직하게 잴 수 있어요
- 파일 이름은 `team_jinsang.csv` 로. 없으면 이 셀은 건너뛰어도 됩니다

In [ ]:
from google.colab import files
up = files.upload()   # team_jinsang.csv 선택
for name in up:
    if name != "team_jinsang.csv":
        import os; os.rename(name, "team_jinsang.csv")
print("업로드 완료")

## 4. 학습 스크립트

In [ ]:
%%writefile train_text_model.py
# ─────────────────────────────────────────────────────────────
# 진상 발언 감지 AI 학습 스크립트
#  - 입력: 음성인식으로 받은 한국어 문장
#  - 출력: "공격적인 말일 확률" (0~1)
#  - 방식: 글자 조각(1~3글자 n-gram) TF-IDF + 로지스틱 회귀
#          → 워치에서 라이브러리 없이 Kotlin 으로 바로 계산 가능
#  - 결과 파일: text_model.tsv (워치 app/src/main/assets 에 넣기)
# ─────────────────────────────────────────────────────────────
import csv, re, math, random, json, sys, os
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score

random.seed(42)
DATA = os.environ.get("DATA_DIR", "data")
SEED_CSV = os.environ.get("SEED_CSV", "jinsang_seed.csv")
OUT = os.environ.get("OUT", "text_model.tsv")

SEED_REPEAT = 15         # 매장 진상 문장은 적으니까 여러 번 반복해서 비중을 키움
MAX_FEATURES = 60000     # 글자 조각 최대 개수 (모델 크기)
NGRAM = (1, 3)

# 워치(Kotlin)와 똑같이 해야 하는 전처리: 소문자 → 한글·영어·숫자만 남기고 띄어쓰기 제거
KEEP = re.compile(r"[^가-힣ㄱ-ㅎㅏ-ㅣa-z0-9]")
def clean(t: str) -> str:
    return KEEP.sub("", t.lower())

def read_tsv(path):
    with open(path, encoding="utf-8") as f:
        return list(csv.reader(f, delimiter="\t", quoting=csv.QUOTE_NONE))

# ── 1. 공개 데이터 ──
def load_unsmile(name):
    rows = read_tsv(f"{DATA}/korean_unsmile_dataset/{name}")
    h = rows[0]
    ci = h.index("clean")
    return [(r[0], 0 if r[ci] == "1" else 1) for r in rows[1:] if len(r) == len(h)]

def load_khs(name):
    rows = read_tsv(f"{DATA}/korean-hate-speech/labeled/{name}")
    return [(r[0], 0 if r[3] == "none" else 1) for r in rows[1:] if len(r) >= 4]

def load_curse():   # 2runo/Curse-detection-data (MIT): "문장|1(욕설)/0"
    path = f"{DATA}/Curse-detection-data/dataset.txt"
    if not os.path.exists(path): return []
    out = []
    for line in open(path, encoding="utf-8"):
        if "|" not in line: continue
        t, l = line.rsplit("|", 1)
        if l.strip() in ("0", "1"): out.append((t, int(l.strip())))
    return out

def load_malicious():   # ZIZUN/korean-malicious-comments-dataset (MIT): 0=악성, 1=정상
    path = f"{DATA}/korean-malicious-comments-dataset/Dataset.csv"
    if not os.path.exists(path): return []
    out = []
    for line in open(path, encoding="utf-8-sig").read().splitlines()[1:]:
        if "\t" not in line: continue
        t, l = line.rsplit("\t", 1)
        if l.strip() in ("0", "1"): out.append((t, 1 - int(l.strip())))
    return out

# 매장 진상 문장: 기본 시드 + 팀이 모은 문장(team_jinsang.csv, 있으면 자동 포함)
# CSV 열: text,label,split   (label 1=진상 0=일반, split 은 train 또는 test, 비우면 train)
TEAM_CSV = os.environ.get("TEAM_CSV", "team_jinsang.csv")
def load_seed(split):
    out = []
    for path in [SEED_CSV, TEAM_CSV]:
        if not os.path.exists(path):
            continue
        with open(path, encoding="utf-8-sig") as f:
            for r in csv.DictReader(f):
                if not r.get("text", "").strip():
                    continue
                if (r.get("split") or "train").strip() == split:
                    out.append((r["text"].strip(), int(r["label"])))
    return out

train = []
train += load_unsmile("unsmile_train_v1.0.tsv")
train += load_khs("train.tsv")
extra = load_curse() + load_malicious()   # 추가 공개 데이터: 10%는 평가용으로 떼어둠
random.shuffle(extra)
cut = len(extra) // 10
train += extra[cut:]
seed_train = load_seed("train")
train += seed_train * SEED_REPEAT

tests = {
    "UnSmile 검증셋 (온라인 댓글)": load_unsmile("unsmile_valid_v1.0.tsv"),
    "Korean HateSpeech 검증셋 (뉴스 댓글)": load_khs("dev.tsv"),
    "욕설·악성댓글 추가 데이터 평가용 10%": extra[:cut],
    "매장 진상 시나리오 (직접 작성, 학습에 안 씀)": load_seed("test"),
}

X = [clean(t) for t, _ in train]
y = [l for _, l in train]
print(f"학습 문장 {len(X)}개 (공격 {sum(y)} / 일반 {len(y) - sum(y)})")

# ── 2. 학습 ──
vec = TfidfVectorizer(analyzer="char", ngram_range=NGRAM, lowercase=False,
                      max_features=MAX_FEATURES, min_df=2, sublinear_tf=False, norm="l2")
Xv = vec.fit_transform(X)
clf = LogisticRegression(C=4.0, max_iter=3000, class_weight="balanced")
clf.fit(Xv, y)

def proba(texts):
    return clf.predict_proba(vec.transform([clean(t) for t in texts]))[:, 1]

# ── 3. 기준값: 매장 일반 대화를 공격으로 잘못 잡지 않는 선에서 정함 ──
seed_neg = [t for t, l in seed_train if l == 0]
neg_p = sorted(proba(seed_neg))
THRESHOLD = round(max(0.5, neg_p[int(len(neg_p) * 0.95)] + 0.02), 2)
print(f"\n판단 기준값(threshold) = {THRESHOLD}")

# ── 4. 평가 ──
report = {}
for name, data in tests.items():
    texts = [t for t, _ in data]; labels = [l for _, l in data]
    pred = [1 if p >= THRESHOLD else 0 for p in proba(texts)]
    r = dict(n=len(data), acc=accuracy_score(labels, pred), f1=f1_score(labels, pred),
             precision=precision_score(labels, pred), recall=recall_score(labels, pred))
    report[name] = r
    print(f"\n[{name}] 문장 {r['n']}개")
    print(f"  정확도 {r['acc']:.3f} · 정밀도 {r['precision']:.3f} · 재현율 {r['recall']:.3f} · F1 {r['f1']:.3f}")

# 매장 시나리오 하나하나 결과 (발표 자료용)
print("\n[매장 시나리오 문장별 결과]")
seed_test = tests["매장 진상 시나리오 (직접 작성, 학습에 안 씀)"]
for (t, l), p in zip(seed_test, proba([t for t, _ in seed_test])):
    mark = "O" if (p >= THRESHOLD) == (l == 1) else "X"
    print(f"  {mark} {'진상' if l else '일반'} {p:5.2f}  {t}")

# ── 5. 워치용 파일로 저장 ──
# 형식: 첫 줄 = 메타 정보(JSON), 이후 한 줄에 "글자조각 \t idf \t 가중치"
names = vec.get_feature_names_out()
idf = vec.idf_
w = clf.coef_[0]
meta = dict(version=1, ngram_min=NGRAM[0], ngram_max=NGRAM[1], bias=float(clf.intercept_[0]),
            threshold=THRESHOLD, features=len(names))
with open(OUT, "w", encoding="utf-8") as f:
    f.write(json.dumps(meta, ensure_ascii=False) + "\n")
    for n, i, c in zip(names, idf, w):
        f.write(f"{n}\t{i:.5f}\t{c:.5f}\n")
print(f"\n저장: {OUT} ({os.path.getsize(OUT) / 1024:.0f} KB, 글자 조각 {len(names)}개)")

# 가장 "공격적"으로 보는 글자 조각 (설명용)
top = sorted(zip(names, w), key=lambda x: -x[1])[:25]
print("\n공격적으로 보는 글자 조각 상위:", " ".join(n for n, _ in top))

json.dump(report, open("eval_report.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)


## 5. 학습 + 평가 (1분 안쪽)

결과의 **매장 진상 시나리오** 정확도와 문장별 O/X가 발표 자료에 쓸 숫자예요.

In [ ]:
!python train_text_model.py

## 6. 워치용 파일 다운로드

받은 `text_model.tsv`를 Android Studio의 `app/src/main/assets/` 에 덮어쓰고 워치에 다시 설치하세요.

In [ ]:
from google.colab import files
files.download("text_model.tsv")
files.download("eval_report.json")

## 7. (선택) 문장 직접 넣어보기

In [ ]:
import importlib.util, sys
exec(open("train_text_model.py", encoding="utf-8").read().split("# ── 5. 워치용 파일로 저장 ──")[0].replace("print(", "(lambda *a, **k: None)("))
for t in ["알바 주제에 어디서 말대꾸야", "봉투 하나만 주세요", "사장 나오라고 해", "배고파 죽겠다"]:
    print(f"{proba([t])[0]:.2f}  {t}")